In [0]:
from pyspark.sql.functions import col, sum, count, when, round, lower, max, abs, try_divide
from pyspark.sql.window import Window

SILVER_SALES_TABLE = "workspace.retail.silver_sales"
GOLD_DIM_PRODUCT = "workspace.retail.gold_dim_product"

def build_product_dimension(df_sales):    
    desc_window = Window.partitionBy("stock_code").orderBy("invoice_date")
    
    df_with_latest_desc = df_sales.withColumn(
        "latest_description", 
        last("description", ignorenulls=True).over(desc_window)
    )
    
    df_products = df_with_latest_desc.groupBy("stock_code").agg(
        max("latest_description").alias("description"),
        
        sum(when(col("quantity") > 0, col("quantity")).otherwise(0)).alias("total_units_sold"),
        sum(when(col("quantity") > 0, col("total_amount")).otherwise(0)).alias("total_revenue"),
        
        sum(when(col("quantity") < 0, abs(col("quantity"))).otherwise(0)).alias("total_units_returned"),
        sum(when(col("quantity") < 0, abs(col("total_amount"))).otherwise(0)).alias("total_refunded_amount")
    )
    
    df_final = df_products.withColumn(
        "return_rate_pct", 
        round(try_divide(col("total_units_returned"), col("total_units_sold")) * 100, 2)
    ) \
        .withColumn(
        "product_status",
        when(col("return_rate_pct") > 10, "High Return Rate")
        .when(col("total_revenue") > 5000, "Best Seller")
        .otherwise("Standard")
    )
    
    return df_final.fillna({"return_rate_pct": 0})

def main():
    df_sales = spark.table(SILVER_SALES_TABLE)
    product_dim_df = build_product_dimension(df_sales)
    
    product_dim_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_DIM_PRODUCT)
        
    print(f"Success! Dimension created with {product_dim_df.count()} unique products.")

if __name__ == "__main__":
    main()